# Encoding Maps Generation

## 1. Setup

In [11]:
import os
import json
import numpy as np
import pandas as pd
raw_path="../Data/Raw Data/raw_data.jsonl"
chunk_size=100000
smoothing=10
date_cutoff="2023-05-18"
os.makedirs('models',exist_ok=True)
cat_map={
    "area":"area_name_en",
    "project":"project_name_en",
    "master_project":"master_project_en",
    "prop_sb_type":"property_sub_type_en",
    "nearest_metro":"nearest_metro_en",
    "nearest_mall":"nearest_mall_en",
    "nearest_landmark":"nearest_landmark_en"
}
needed_cols=list(cat_map.values())+["instance_date","trans_group_en","meter_sale_price"]

## 2. Streaming aggregation

The raw file was too big. Since that is the case, the code processes the data in 100,000 chunks

In [13]:
sums={clean_col:{} for clean_col in cat_map}
counts={clean_col:{} for clean_col in cat_map}
global_sum=0.0
global_count=0
rows_processed=0
rows_kept=0
reader = pd.read_json(raw_path, lines=True, chunksize=chunk_size)
for chunk_idx, chunk in enumerate(reader, start=1):
    rows_processed+=len(chunk)
    chunk=chunk[[c for c in needed_cols if c in chunk.columns]]
    chunk["instance_date"]=pd.to_datetime(chunk["instance_date"],errors="coerce")
    chunk=chunk.dropna(subset=["instance_date"])
    chunk=chunk[chunk["instance_date"]>=date_cutoff]
    chunk=chunk[chunk["trans_group_en"]=="Sales"]
    chunk=chunk[chunk["meter_sale_price"]>0]
    chunk=chunk.dropna(subset=["meter_sale_price"])
    if len(chunk)==0:
        if chunk_idx%5==0:
            print(f"Chunk {chunk_idx}: processed {rows_processed} rows and kept {rows_kept}")
        continue
    rows_kept+=len(chunk)
    for raw_col in cat_map.values():
        if raw_col in chunk.columns:
            chunk[raw_col]=chunk[raw_col].fillna("Unknown")
    global_sum+=float(chunk["meter_sale_price"].sum())
    global_count+=len(chunk)
    for clean_col,raw_col in cat_map.items():
        if raw_col not in chunk.columns:
            continue
        grp=chunk.groupby(raw_col)["meter_sale_price"].agg(["sum","count"])
        for cat,row in grp.iterrows():
            sums[clean_col][cat]=sums[clean_col].get(cat, 0.0) + float(row["sum"])
            counts[clean_col][cat]=counts[clean_col].get(cat,0)+int(row["count"])
    if chunk_idx%5==0:
        print(f"  Chunk {chunk_idx}: processed {rows_processed} rows and kept {rows_kept}")

  Chunk 5: processed 500000 rows and kept 160130
  Chunk 10: processed 1000000 rows and kept 320510
  Chunk 15: processed 1500000 rows and kept 480442


## 3. Build the smoothed encoding maps

In [14]:
if global_count==0:
    raise ValueError("No rows survived the filters.")
global_mean=global_sum/global_count
print(f"Global mean meter_sale_price: {global_mean} AED/m^2")
print(f"Total rows used for encoding: {global_count}")
print()
encoding_maps={}
for clean_col in cat_map:
    encoding_maps[clean_col]={}
    for cat,cat_sum in sums[clean_col].items():
        n=counts[clean_col][cat]
        mean=cat_sum/n
        smoothed=(mean*n+global_mean*smoothing) / (n+smoothing)
        encoding_maps[clean_col][str(cat)]=float(smoothed)

Global mean meter_sale_price: 19502.974476657353 AED/m^2
Total rows used for encoding: 550366



## 4. Save the maps to disk

In [15]:
with open("models/encoding_maps.json","w") as f:
    json.dump(encoding_maps,f,indent=2)
with open("models/global_means.json","w") as f:
    json.dump({"price_per_sqm_mean": global_mean,"fallback_for_unseen_category": global_mean,"rows_used": global_count},
    f,indent=2)